# BEATs Core-2 N/A/L Ablation — Seed 42

**Status: READY_FOR_USER_START.** This notebook prepares the meeting-approved sequence: waveform normalization, then waveform augmentation, then loss. It does not run unless `RUN=True` is set after explicit approval.

## Frozen comparison contract

ICBHI + SPRSound only; shared Level1/Crackle/Wheeze hierarchy; seed 42; mono 16 kHz; native-unit normalization/augmentation before 2 s windows with 1 s stride; effective native batch 8 via unit-wise gradient accumulation; 50 epochs; Adam; cosine schedule; FP32. Selection is the equal mean of ICBHI and SPRSound validation eligible-node loss. Test is not part of this training notebook. Validation artifacts retain sample ID, group ID, filename, raw ground truth, unified targets/eligibility, logits, probabilities, and predictions.

## Staged sequence

1. `R0`: full BEATs, raw waveform, no augmentation, CE+BCE reference.
2. `N1`: change only native-waveform peak normalization.
3. `A1`: keep the selected normalization and add seeded gain + additive-noise augmentation.
4. `L1/L2`: keep the selected waveform policy and compare focal versus effective-number class-balanced loss.

The frozen-encoder mode is retained as an engineering/attribution option, not as the main SOTA-facing run. RMS normalization is supported only when an explicit training-derived dBFS target is supplied.

## Literature-facing readout

The selected validation predictions choose one core-shared Crackle threshold and one core-shared Wheeze threshold by equal ICBHI/SPRSound F1. ICBHI flat4 is reconstructed only from the two atomic bits: `00 Normal`, `10 Crackle`, `01 Wheeze`, `11 Both`; Level1 is reported separately and never overrides flat4. SPRSound Task1-1 uses Level1. Task1-2 raw7 remains a native reference and is not claimed as an output of this shared head.

In [ ]:
from pathlib import Path

from baseline.multidataset_pipeline.beats_nal_protocol import (
    BEATsNALConfig, HierarchicalLossConfig, WaveformAugmentationConfig,
    WaveformNormalizationConfig, run_training,
)

ROOT = Path.cwd()
SOURCE = ROOT / '.cache/multidataset_pipeline/assets/P2/source/repo'
CHECKPOINT = ROOT / '.cache/multidataset_pipeline/assets/P2/checkpoints/BEATs_iter3_plus_AS2M.pt'
RUN = False  # Change only after explicit user approval.
CONDITION = 'R0'

CONDITIONS = {
    'R0': dict(normalization='none', augmentation='none', loss='ce_bce'),
    'N1': dict(normalization='peak', augmentation='none', loss='ce_bce'),
    'A1': dict(normalization='peak', augmentation='gain_noise', loss='ce_bce'),
    'L1': dict(normalization='peak', augmentation='gain_noise', loss='focal'),
    'L2': dict(normalization='peak', augmentation='gain_noise', loss='class_balanced'),
}
choice = CONDITIONS[CONDITION]
config = BEATsNALConfig(
    repo_root=ROOT,
    source_repo=SOURCE,
    checkpoint=CHECKPOINT,
    output_dir=ROOT / 'result/reproduce/beats_nal_ablation' / CONDITION,
    device='cpu',
    encoder_scope='full',
    normalization=WaveformNormalizationConfig(mode=choice['normalization']),
    augmentation=WaveformAugmentationConfig(mode=choice['augmentation']),
    loss=HierarchicalLossConfig(mode=choice['loss']),
)
status = run_training(config) if RUN else {
    'status': 'READY_FOR_USER_START',
    'condition': CONDITION,
    'config': config.to_dict(),
}
status

## Evidence boundary

A completed subtrain/validation run is not a terminal result. The selected checkpoint may access official test only through a separately approved terminal step. Package results are single-seed controlled observations and must not be described as a pure backbone ranking or SOTA reproduction.